# full_analysis.ipynb — RQ1 Statistical Analysis
**Nhóm 3 — SWT302** | Qwen2.5-7B-Instruct 3-shot vs GPT-4o threshold (E02)

> **Cách chạy:** Restart Kernel → Run All. Đảm bảo `full_metrics_output_FINAL.csv` cùng thư mục hoặc chỉnh `DATA_PATH` bên dưới.

In [ ]:
# Cell 1 — Cài thư viện
# !pip install pandas numpy scipy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import wilcoxon
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded OK')

In [ ]:
# Cell 2 — Load data
# Chỉnh đường dẫn nếu cần
DATA_PATH = 'full_metrics_output_FINAL.csv'

df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape[0]} rows x {df.shape[1]} cols')
print(f'Columns: {list(df.columns)}')
print(f'Null values:\n{df.isnull().sum()}')
assert df.isnull().sum().sum() == 0, 'ERROR: có giá trị null!'
print('\nOK — Không có null')

## 1. Descriptive Statistics (Thống kê mô tả)

In [ ]:
# Cell 3 — Thống kê mô tả
metrics_cols = ['ctqrs_percent', 'rouge1_f1', 'sbert_similarity']
desc = df[metrics_cols].describe().T
desc.index = ['CTQRS (%)', 'ROUGE-1 F1', 'SBERT']
print(f'N = {len(df)} samples')
print()
print(desc[['mean','std','min','25%','50%','75%','max']].round(4).to_string())

# Thresholds tham chiếu E02 GPT-4o 3-shot
thresholds = {'CTQRS (%)': 75.0, 'ROUGE-1 F1': 0.47, 'SBERT': 0.82}
print('\nThresholds (E02 GPT-4o 3-shot):')
for k, v in thresholds.items():
    print(f'  {k}: {v}')

## 2. Wilcoxon Signed-Rank One-Sample Test (α = 0.05)

In [ ]:
# Cell 4 — Wilcoxon + Cliff's Delta

def cliffs_delta(data, threshold):
    data = np.array(data)
    return (np.sum(data > threshold) - np.sum(data < threshold)) / len(data)

def interpret_cliffs(d):
    a = abs(d)
    if a < 0.147: return 'negligible'
    elif a < 0.33: return 'small'
    elif a < 0.474: return 'medium'
    else: return 'large'

def wilcoxon_one_sample(data, threshold, name):
    diff = np.array(data) - threshold
    diff_nz = diff[diff != 0]
    stat, p = wilcoxon(diff_nz, alternative='greater')
    med = np.median(data)
    d = cliffs_delta(data, threshold)
    reject = (p < 0.05) and (med >= threshold)
    print(f'{name}')
    print(f'  median={med:.4f} | threshold={threshold} | p={p:.6f} | delta={d:.3f} ({interpret_cliffs(d)}) | {"ĐẠT" if reject else "KHÔNG ĐẠT"}')
    return dict(metric=name, N=len(data), median=round(med,4), threshold=threshold,
                p_value=round(p,6), p_bonferroni=round(p*3,6),
                cliffs_delta=round(d,3), effect_size=interpret_cliffs(d),
                reject_H0=reject, conclusion='ĐẠT ngưỡng' if reject else 'KHÔNG ĐẠT ngưỡng')

print('=== Wilcoxon Signed-Rank One-Sample Test (RQ1, α=0.05) ===')
r1 = wilcoxon_one_sample(df['ctqrs_percent'],    75.0, 'H1.1 CTQRS (≥75%)')
r2 = wilcoxon_one_sample(df['sbert_similarity'],  0.82, 'H1.2 SBERT (≥0.82)')
r3 = wilcoxon_one_sample(df['rouge1_f1'],         0.47, 'H1.3 ROUGE-1 (≥0.47)')
alpha_bonf = 0.05 / 3
print(f'\nBonferroni α_adjusted = 0.05/3 = {alpha_bonf:.4f}')

## 3. Summary Table

In [ ]:
# Cell 5 — Tạo summary DataFrame + lưu summary.csv
results = [r1, r2, r3]
summary_df = pd.DataFrame(results)
summary_df.insert(0, 'RQ', 'RQ1')
summary_df.insert(1, 'model', 'Qwen2.5-7B-Instruct')
summary_df.insert(2, 'prompt_strategy', '3-shot Alpaca-LoRA')
summary_df.insert(3, 'dataset', 'Bugzilla GindeLab 10% seed=42')

print(summary_df[['metric','N','median','threshold','p_value','p_bonferroni',
                   'cliffs_delta','effect_size','conclusion']].to_string(index=False))

summary_df.to_csv('summary.csv', index=False)
print('\nSaved: summary.csv')

## 4. Figures

In [ ]:
# Cell 6 — Figure 1: Boxplot (3 metrics vs threshold)
fig, axes = plt.subplots(1, 3, figsize=(14, 6))
fig.suptitle('RQ1: Qwen2.5-7B 3-shot vs E02 Thresholds (N=262)', fontsize=14, fontweight='bold')

configs = [
    ('ctqrs_percent',    'CTQRS (%)',   75.0,  '#4C72B0'),
    ('rouge1_f1',        'ROUGE-1 F1',  0.47,  '#55A868'),
    ('sbert_similarity', 'SBERT',       0.82,  '#C44E52'),
]

for ax, (col, label, thresh, color) in zip(axes, configs):
    data = df[col].values
    bp = ax.boxplot(data, patch_artist=True, widths=0.5,
                    medianprops=dict(color='black', linewidth=2))
    bp['boxes'][0].set_facecolor(color)
    bp['boxes'][0].set_alpha(0.7)
    ax.axhline(thresh, color='red', linestyle='--', linewidth=1.5, label=f'Threshold = {thresh}')
    med = np.median(data)
    ax.set_title(f'{label}\nMedian={med:.4f}', fontsize=11)
    ax.set_ylabel(label)
    ax.set_xticks([])
    ax.legend(fontsize=9)
    ax.annotate(f'N={len(data)}', xy=(0.95,0.02), xycoords='axes fraction',
                ha='right', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig('figure1_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figure1_boxplot.png')

In [ ]:
# Cell 7 — Figure 2: Distribution histogram (3 metrics)
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Distribution of Metrics — Qwen2.5-7B 3-shot (N=262)', fontsize=13, fontweight='bold')

for ax, (col, label, thresh, color) in zip(axes, configs):
    data = df[col].values
    ax.hist(data, bins=25, color=color, alpha=0.75, edgecolor='white')
    ax.axvline(thresh, color='red', linestyle='--', linewidth=1.8, label=f'Threshold={thresh}')
    ax.axvline(np.median(data), color='navy', linestyle='-', linewidth=1.5,
               label=f'Median={np.median(data):.3f}')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.set_title(label)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('figure2_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figure2_distribution.png')

## 5. Kết luận RQ1

In [ ]:
# Cell 8 — Print kết luận
print('='*60)
print('KẾT QUẢ RQ1 — MIXED')
print('='*60)
print()
print('RQ1: Qwen2.5-7B-Instruct 3-shot có đạt CTQRS≥75%,')
print('     SBERT≥0.82, ROUGE-1≥0.47 (ngưỡng GPT-4o E02) không?')
print()
for r in [r1, r2, r3]:
    icon = '✅' if r['reject_H0'] else '❌'
    print(f"{icon} {r['metric']}: median={r['median']} vs threshold={r['threshold']} "
          f"| p={r['p_value']} | δ={r['cliffs_delta']} ({r['effect_size']}) "
          f"→ {r['conclusion']}")
print()
print('Diễn giải:')
print('  CTQRS cao (88.9%) → Qwen tạo report có cấu trúc tốt')
print('  ROUGE-1 đạt (0.491) → Overlap từ vựng tương đương GPT-4o')
print('  SBERT thấp (0.632) → Style mismatch: Qwen dài hơn GT 1.82x,')
print('    văn phong khác biệt với ground truth Bugzilla')
print()
print(f'Bonferroni α = 0.05/3 = {0.05/3:.4f}')
print('Kết quả không thay đổi sau Bonferroni correction.')